In [1]:
import scanpy as sc
import anndata
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import scvi
import seaborn as sb

In [2]:
os.chdir("/home/lixiangyu/zr/Annotate/")

#### output data:/home/lixiangyu/zr/Annotate/output_data/public_data/new0509/merged.h5ad

# TAA

## GSE143921 6759 6058

In [3]:
file_paths = [
    "data/public_data/GSE143921/TAA_GSE143921_101932_AAD/TAA_GSM143921_101932_AAD.h5ad",
    "data/public_data/GSE143921/TAA_GSE143921_103973/TAA_GSE143921_103973.h5ad",
    "data/public_data/GSE143921/TAA_GSE143921_104021/TAA_GSE143921_104021.h5ad",
    "data/public_data/GSE143921/TAA_GSE143921_104041/TAA_GSE143921_104041.h5ad",
    "data/public_data/GSE143921/TAA_GSE143921_113508/TAA_GSE143921_113508.h5ad",
    "data/public_data/GSE143921/TAA_GSE143921_113557_aaa/TAA_GSE143921_113557_aaa.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]


In [4]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)


Number of cells in sample Sample1: 847
Number of cells in sample Sample2: 1227
Number of cells in sample Sample3: 1069
Number of cells in sample Sample4: 1070
Number of cells in sample Sample5: 1377
Number of cells in sample Sample6: 1169


In [5]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

adata_final.write_h5ad("data/public_data/GSE143921/TAA_GSE143921_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata_final = sc.read_h5ad("data/public_data/GSE143921/TAA_GSE143921_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
adata_final

AnnData object with n_obs × n_vars = 6759 × 52117
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [8]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_442781/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [ ]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE143921/TAA_GSE143921_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE143921/TAA_GSE143921_merged_postR.h5ad")

In [ ]:
adata_final = sc.read_h5ad("data/public_data/GSE143921/TAA_GSE143921_merged_postR.h5ad")

In [ ]:
adata_final

AnnData object with n_obs × n_vars = 6759 × 52117
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE143921_1_UMAP', 'decontX_GSE143921_2_UMAP', 'decontX_GSE143921_3_UMAP', 'decontX_GSE143921_4_UMAP', 'decontX_GSE143921_5_UMAP', 'decontX_GSE143921_6_UMAP'
    layers: 'decontXcounts'

In [ ]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100) 
sc.pp.filter_cells(adata_final, max_genes=8000) 
sc.pp.filter_genes(adata_final, min_cells=3)
print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 6759
Number of cells before counts filter: 6425
Number of cells beforeMT filter: 6058
Number of cells after MT filter: 6058


In [ ]:
adata_final.write("data/public_data/GSE143921/TAA_GSE143921_/var/folders/nt/yxtswcqn7j9_8lq5g91x39bh0000gp/T/FB8B589D-8D6E-4AD9-85ED-EADEF3F68A36.pngpostQC-R.h5ad")

In [3]:
adata_final = sc.read_h5ad("data/public_data/GSE143921/TAA_GSE143921_postQC-R.h5ad")

In [50]:
adata_final.obs['sample'].value_counts()

sample
GSE143921_6    1289
GSE143921_2    1160
GSE143921_4    1052
GSE143921_3    1034
GSE143921_5    1008
GSE143921_1     515
Name: count, dtype: int64

## GSE155468 48128 44658/47271

In [70]:
file_paths = [
    "data/public_data/GSE155468/TAA_GSE155468_healthy4/TAA_GSE155468_healthy4.h5ad",
    "data/public_data/GSE155468/TAA_GSE155468_healthy6/TAA_GSE155468_healthy6.h5ad",
    "data/public_data/GSE155468/TAA_GSE155468_healthy9/TAA_GSE155468_healthy9.h5ad",
    "data/public_data/GSE155468/TAA1_GSE155468/TAA1_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA2_GSE155468/TAA2_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA3_GSE155468/TAA3_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA4_GSE155468/TAA4_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA5_GSE155468/TAA5_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA6_GSE155468/TAA6_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA7_GSE155468/TAA7_GSE155468.h5ad",
    "data/public_data/GSE155468/TAA8_GSE155468/TAA8_GSE155468.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10",
    "Sample11"
]

In [71]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 3377
Number of cells in sample Sample2: 1193
Number of cells in sample Sample3: 3907
Number of cells in sample Sample4: 6686
Number of cells in sample Sample5: 4598
Number of cells in sample Sample6: 3622
Number of cells in sample Sample7: 6384
Number of cells in sample Sample8: 4802
Number of cells in sample Sample9: 3526
Number of cells in sample Sample10: 6997
Number of cells in sample Sample11: 3036


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [72]:
adata_final.write_h5ad("data/public_data/GSE155468/TAA_GSE155468_merged.h5ad")

In [73]:
adata_final

AnnData object with n_obs × n_vars = 48128 × 23331
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [51]:
adata_final = sc.read_h5ad("data/public_data/GSE155468/TAA_GSE155468_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [52]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [54]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE155468/TAA_GSE155468_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE155468/TAA_GSE155468_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 16:07:30 2025 .. Analyzing cells in batch 'GSE155468_1'
Tue Nov 25 16:07:31 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 16:07:46 2025 .... Estimating contamination
Tue Nov 25 16:07:48 2025 ...... Completed iteration: 10 | converge: 0.1173
Tue Nov 25 16:07:50 2025 ...... Completed iteration: 20 | converge: 0.009344
Tue Nov 25 16:07:52 2025 ...... Completed iteration: 30 | converge: 0.005768
Tue Nov 25 16:07:54 2025 ...... Completed iteration: 40 | converge: 0.003345
Tue Nov 25 16:07:56 2025 ...... Completed iteration: 50 | converge: 0.002828
Tue Nov 25 16:07:58 2025 ...... Completed iteration: 60 | converge: 0.001424
Tue Nov 25 16:07:59 2025 ...... Completed iteration: 69 | converge: 0.000963
Tue Nov 25 16:07:59 2025 .. Calculating final decontaminated matrix
Tue Nov 25 16:08:01 2025 .. Analyzing cells in batch 'GSE155468_2'
Tue Nov 25 16:0

In [55]:
adata_final = sc.read_h5ad("data/public_data/GSE155468/TAA_GSE155468_merged_postR.h5ad")

In [56]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [57]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=4000) 
# sc.pp.filter_genes(adata_final, min_cells=3) 

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
# sc.pp.filter_cells(adata_final, max_counts = 30000)

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 48128
Number of cells beforeMT filter: 47271
Number of cells after MT filter: 47271


In [58]:
adata_final.write("data/public_data/GSE155468/TAA_GSE155468_postQC-R.h5ad")

In [59]:
adata_final = sc.read_h5ad("data/public_data/GSE155468/TAA_GSE155468_postQC-R.h5ad")

In [60]:
adata_final.obs['sample'].value_counts()

sample
GSE155468_4     6686
GSE155468_10    6471
GSE155468_7     6384
GSE155468_8     4802
GSE155468_5     4598
GSE155468_3     3907
GSE155468_6     3622
GSE155468_9     3404
GSE155468_1     3168
GSE155468_11    3036
GSE155468_2     1193
Name: count, dtype: int64

## GSE241870

# AAA

## GSE226492 139806 134125/96108

In [79]:
file_paths = [
    "data/public_data/GSE226492_raw/AAA_GSE226492_healthy1/AAA_GSE226492_healthy1.h5ad",
    "data/public_data/GSE226492_raw/AAA_GSE226492_healthy2/AAA_GSE226492_healthy2.h5ad",
    "data/public_data/GSE226492_raw/AAA_GSE226492_healthy3/AAA_GSE226492_healthy3.h5ad",
    "data/public_data/GSE226492_raw/AAA1_GSE226492/AAA1_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/AAA2_GSE226492/AAA2_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/AAA3_GSE226492/AAA3_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/AD1_GSE226492/AD1_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/AD2_GSE226492/AD2_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/AD3_GSE226492/AD3_GSE226492.h5ad",
    "data/public_data/GSE226492_raw/MF_GSE226492/MF_GSE226492.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10"
]

In [80]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 11039
Number of cells in sample Sample2: 15811
Number of cells in sample Sample3: 19605
Number of cells in sample Sample4: 13424
Number of cells in sample Sample5: 7494
Number of cells in sample Sample6: 9786
Number of cells in sample Sample7: 10658
Number of cells in sample Sample8: 12951
Number of cells in sample Sample9: 18616
Number of cells in sample Sample10: 20422


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [81]:
adata_final.write_h5ad("data/public_data/GSE226492_raw/AAA_GSE226492_merged.h5ad")

In [82]:
adata_final

AnnData object with n_obs × n_vars = 139806 × 32738
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [61]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [62]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE226492_raw/AAA_GSE226492_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE226492_raw/AAA_GSE226492_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 16:49:34 2025 .. Analyzing cells in batch 'GSE226492_1'
Tue Nov 25 16:49:35 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 16:50:06 2025 .... Estimating contamination
Tue Nov 25 16:50:13 2025 ...... Completed iteration: 10 | converge: 0.02908
Tue Nov 25 16:50:19 2025 ...... Completed iteration: 20 | converge: 0.01898
Tue Nov 25 16:50:25 2025 ...... Completed iteration: 30 | converge: 0.009075
Tue Nov 25 16:50:31 2025 ...... Completed iteration: 40 | converge: 0.005794
Tue Nov 25 16:50:37 2025 ...... Completed iteration: 50 | converge: 0.005535
Tue Nov 25 16:50:43 2025 ...... Completed iteration: 60 | converge: 0.005131
Tue Nov 25 16:50:50 2025 ...... Completed iteration: 70 | converge: 0.004856
Tue Nov 25 16:50:56 2025 ...... Completed iteration: 80 | converge: 0.005025
Tue Nov 25 16:51:02 2025 ...... Completed iteration: 90 | converge: 0.005

In [63]:
adata_final = sc.read_h5ad("data/public_data/GSE226492_raw/AAA_GSE226492_merged_postR.h5ad")

In [64]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [65]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=7000) # acc to author
# sc.pp.filter_genes(adata_final, min_cells=3)

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

# sc.pp.filter_cells(adata_final, max_counts = 200000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 139806
Number of cells beforeMT filter: 138975
Number of cells after MT filter: 96108


In [66]:
adata_final.write("data/public_data/GSE226492_raw/AAA_GSE226492_postQC-R.h5ad")

In [67]:
adata_final = sc.read_h5ad("data/public_data/GSE226492_raw/AAA_GSE226492_postQC-R.h5ad")

In [68]:
adata_final

AnnData object with n_obs × n_vars = 96108 × 32738
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE226492_10_UMAP', 'decontX_GSE226492_1_UMAP', 'decontX_GSE226492_2_UMAP', 'decontX_GSE226492_3_UMAP', 'decontX_GSE226492_4_UMAP', 'decontX_GSE226492_5_UMAP', 'decontX_GSE226492_6_UMAP', 'decontX_GSE226492_7_UMAP', 'decontX_GSE226492_8_UMAP', 'decontX_GSE226492_9_UMAP'
    layers: 'decontXcounts'

In [69]:
adata_final.obs['sample'].value_counts()

sample
GSE226492_10    17356
GSE226492_9     15853
GSE226492_2     12777
GSE226492_8     10748
GSE226492_3     10602
GSE226492_1      9069
GSE226492_7      6636
GSE226492_4      5270
GSE226492_6      4499
GSE226492_5      3298
Name: count, dtype: int64

## GSE224587 85149 82055 

In [73]:
file_paths = [
    "data/public_data/GSE224587_raw/AAA1_GSE224587_center/AAA1_GSE224587_center.h5ad",
    "data/public_data/GSE224587_raw/AAA1_GSE224587_prox/AAA1_GSE224587_prox.h5ad",
    "data/public_data/GSE224587_raw/AAA2_GSE224587_center/AAA2_GSE224587_center.h5ad",
    "data/public_data/GSE224587_raw/AAA2_GSE224587_prox/AAA2_GSE224587_prox.h5ad",
    "data/public_data/GSE224587_raw/AAA3_GSE224587_center/AAA3_GSE224587_center.h5ad",
    "data/public_data/GSE224587_raw/AAA3_GSE224587_prox/AAA3_GSE224587_prox.h5ad",
    "data/public_data/GSE224587_raw/AAA4_GSE224587_center/AAA4_GSE224587_center.h5ad",
    "data/public_data/GSE224587_raw/AAA5_GSE224587_center/AAA5_GSE224587_center.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8"
]

In [74]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 13649
Number of cells in sample Sample2: 6973
Number of cells in sample Sample3: 20544
Number of cells in sample Sample4: 1879
Number of cells in sample Sample5: 10069
Number of cells in sample Sample6: 2804
Number of cells in sample Sample7: 20753
Number of cells in sample Sample8: 8478


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [75]:
adata_final.write_h5ad("data/public_data/GSE224587_raw/AAA_GSE224587_merged.h5ad")

In [76]:
adata_final

AnnData object with n_obs × n_vars = 85149 × 36601
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [77]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [78]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE224587_raw/AAA_GSE224587_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE224587_raw/AAA_GSE224587_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 17:39:18 2025 .. Analyzing cells in batch 'GSE224587_1'
Tue Nov 25 17:39:18 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 17:39:55 2025 .... Estimating contamination
Tue Nov 25 17:40:02 2025 ...... Completed iteration: 10 | converge: 0.04012
Tue Nov 25 17:40:08 2025 ...... Completed iteration: 20 | converge: 0.01845
Tue Nov 25 17:40:14 2025 ...... Completed iteration: 30 | converge: 0.01403
Tue Nov 25 17:40:19 2025 ...... Completed iteration: 40 | converge: 0.009191
Tue Nov 25 17:40:25 2025 ...... Completed iteration: 50 | converge: 0.005583
Tue Nov 25 17:40:31 2025 ...... Completed iteration: 60 | converge: 0.004805
Tue Nov 25 17:40:37 2025 ...... Completed iteration: 70 | converge: 0.004132
Tue Nov 25 17:40:42 2025 ...... Completed iteration: 80 | converge: 0.003441
Tue Nov 25 17:40:48 2025 ...... Completed iteration: 90 | converge: 0.0028

In [79]:
adata_final = sc.read_h5ad("data/public_data/GSE224587_raw/AAA_GSE224587_merged_postR.h5ad")

In [80]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [81]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100) 
sc.pp.filter_cells(adata_final, max_genes=8000) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts =50000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 85149


Number of cells before counts filter: 84988
Number of cells beforeMT filter: 84827
Number of cells after MT filter: 82055


In [82]:
adata_final.write("data/public_data/GSE224587_raw/AAA_GSE224587_postQC-R.h5ad")

In [83]:
adata_final = sc.read_h5ad("data/public_data/GSE224587_raw/AAA_GSE224587_postQC-R.h5ad")

In [84]:
adata_final.obs['sample'].value_counts()

sample
GSE224587_3    20017
GSE224587_7    19288
GSE224587_1    13477
GSE224587_5     9788
GSE224587_8     8253
GSE224587_2     6802
GSE224587_6     2637
GSE224587_4     1793
Name: count, dtype: int64

## GSE237230 6552 5727/4386

In [96]:
file_paths = [
    "data/public_data/GSE237230_raw/AAA_GSE237230_02257/AAA_GSE237230_02257.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_02258/AAA_GSE237230_02258.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_14320/AAA_GSE237230_14320.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_14322/AAA_GSE237230_14322.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_22896/AAA_GSE237230_22896.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_22897/AAA_GSE237230_22897.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_22898/AAA_GSE237230_22898.h5ad",
    "data/public_data/GSE237230_raw/AAA_GSE237230_22899/AAA_GSE237230_22899.h5ad",
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8"
]

In [97]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 771
Number of cells in sample Sample2: 1528
Number of cells in sample Sample3: 414
Number of cells in sample Sample4: 437
Number of cells in sample Sample5: 562
Number of cells in sample Sample6: 1630
Number of cells in sample Sample7: 224
Number of cells in sample Sample8: 986


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [98]:
adata_final.write_h5ad("data/public_data/GSE237230_raw/AAA_GSE237230_merged.h5ad")

In [99]:
adata_final

AnnData object with n_obs × n_vars = 6552 × 60663
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [85]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [86]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE237230_raw/AAA_GSE237230_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE237230_raw/AAA_GSE237230_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 17:52:34 2025 .. Analyzing cells in batch 'GSE237230_1'
Tue Nov 25 17:52:34 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 17:52:41 2025 .... Estimating contamination
Tue Nov 25 17:52:41 2025 ...... Completed iteration: 10 | converge: 0.02562
Tue Nov 25 17:52:42 2025 ...... Completed iteration: 20 | converge: 0.009341
Tue Nov 25 17:52:42 2025 ...... Completed iteration: 30 | converge: 0.005728
Tue Nov 25 17:52:42 2025 ...... Completed iteration: 40 | converge: 0.004123
Tue Nov 25 17:52:43 2025 ...... Completed iteration: 50 | converge: 0.002003
Tue Nov 25 17:52:43 2025 ...... Completed iteration: 60 | converge: 0.001485
Tue Nov 25 17:52:43 2025 ...... Completed iteration: 70 | converge: 0.001429
Tue Nov 25 17:52:43 2025 ...... Completed iteration: 78 | converge: 0.0009718
Tue Nov 25 17:52:43 2025 .. Calculating final decontaminated matrix
Tue

In [87]:
adata_final = sc.read_h5ad("data/public_data/GSE237230_raw/AAA_GSE237230_merged_postR.h5ad")

In [88]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [89]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  
sc.pp.filter_cells(adata_final, max_genes=4000) 
# sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 20000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 6552
Number of cells before counts filter: 5592
Number of cells beforeMT filter: 5566
Number of cells after MT filter: 4386


In [90]:
adata_final.write("data/public_data/GSE237230_raw/AAA_GSE237230_postQC-R.h5ad")

In [91]:
adata_final = sc.read_h5ad("data/public_data/GSE237230_raw/AAA_GSE237230_postQC-R.h5ad")

In [92]:
adata_final.obs['sample'].value_counts()

sample
GSE237230_6    1321
GSE237230_2     767
GSE237230_8     725
GSE237230_1     580
GSE237230_5     385
GSE237230_3     249
GSE237230_4     214
GSE237230_7     145
Name: count, dtype: int64

## GSE166676 14088 12861/12427

In [105]:
file_paths = [
    "data/public_data/GSE166676_raw/AAA_GSE166676_27_190226/AAA_GSE166676_27_190226.h5ad",
    "data/public_data/GSE166676_raw/AAA_GSE166676_28_190524/AAA_GSE166676_28_190524.h5ad",
    "data/public_data/GSE166676_raw/AAA_GSE166676_29_190524/AAA_GSE166676_29_190524.h5ad",
    "data/public_data/GSE166676_raw/AAA_GSE166676_30_190617/AAA_GSE166676_30_190617.h5ad",
    "data/public_data/GSE166676_raw/AAA_GSE166676_31_190703/AAA_GSE166676_31_190703.h5ad",
    "data/public_data/GSE166676_raw/AAA_GSE166676_32_191127/AAA_GSE166676_32_191127.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]

In [106]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 3100
Number of cells in sample Sample2: 2332
Number of cells in sample Sample3: 2497
Number of cells in sample Sample4: 758
Number of cells in sample Sample5: 2209
Number of cells in sample Sample6: 3192


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [107]:
adata_final.write_h5ad("data/public_data/GSE166676_raw/AAA_GSE166676_merged.h5ad")

In [93]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [94]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/GSE166676_raw/AAA_GSE166676_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/GSE166676_raw/AAA_GSE166676_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 17:55:55 2025 .. Analyzing cells in batch 'GSE166676_1'
Tue Nov 25 17:55:55 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 17:56:09 2025 .... Estimating contamination
Tue Nov 25 17:56:11 2025 ...... Completed iteration: 10 | converge: 0.03547
Tue Nov 25 17:56:12 2025 ...... Completed iteration: 20 | converge: 0.0163
Tue Nov 25 17:56:13 2025 ...... Completed iteration: 30 | converge: 0.0107
Tue Nov 25 17:56:14 2025 ...... Completed iteration: 40 | converge: 0.007446
Tue Nov 25 17:56:15 2025 ...... Completed iteration: 50 | converge: 0.005471
Tue Nov 25 17:56:16 2025 ...... Completed iteration: 60 | converge: 0.00438
Tue Nov 25 17:56:17 2025 ...... Completed iteration: 70 | converge: 0.003331
Tue Nov 25 17:56:18 2025 ...... Completed iteration: 80 | converge: 0.002618
Tue Nov 25 17:56:20 2025 ...... Completed iteration: 90 | converge: 0.002013


In [95]:
adata_final =sc.read_h5ad("data/public_data/GSE166676_raw/AAA_GSE166676_merged_postR.h5ad")

In [96]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [97]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=2500) # acc to author
# sc.pp.filter_genes(adata_final, min_cells=3)

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

# sc.pp.filter_cells(adata_final, max_counts = 20000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 25] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 14088
Number of cells beforeMT filter: 12721
Number of cells after MT filter: 12427


In [98]:
adata_final.write("data/public_data/GSE166676_raw/AAA_GSE166676_postQC-R.h5ad")

In [99]:
adata_final = sc.read_h5ad("data/public_data/GSE166676_raw/AAA_GSE166676_postQC-R.h5ad")

In [100]:
adata_final.obs['sample'].value_counts()

sample
GSE166676_6    3088
GSE166676_1    2601
GSE166676_3    2220
GSE166676_2    2091
GSE166676_5    1883
GSE166676_4     544
Name: count, dtype: int64

# AS_atlas

## GSE155512 8867 8866

In [24]:
file_paths = [
    "data/public_data/AS_atlas/GSE155512/GSM4705589_RPE004/GSM4705589_RPE004.h5ad",
    "data/public_data/AS_atlas/GSE155512/GSM4705590_RPE005/GSM4705590_RPE005.h5ad",
    "data/public_data/AS_atlas/GSE155512/GSM4705591_RPE006/GSM4705591_RPE006.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]

In [25]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 2614
Number of cells in sample Sample2: 3486
Number of cells in sample Sample3: 2767


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE155512/GSE155512_merged.h5ad")

In [27]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [101]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE155512/GSE155512_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE155512/GSE155512_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 18:56:48 2025 .. Analyzing cells in batch 'GSE155512_1'
Tue Nov 25 18:56:48 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 18:57:01 2025 .... Estimating contamination
Tue Nov 25 18:57:02 2025 ...... Completed iteration: 10 | converge: 0.01951
Tue Nov 25 18:57:04 2025 ...... Completed iteration: 20 | converge: 0.007195
Tue Nov 25 18:57:05 2025 ...... Completed iteration: 30 | converge: 0.004442
Tue Nov 25 18:57:06 2025 ...... Completed iteration: 40 | converge: 0.002818
Tue Nov 25 18:57:08 2025 ...... Completed iteration: 50 | converge: 0.001826
Tue Nov 25 18:57:09 2025 ...... Completed iteration: 60 | converge: 0.001225
Tue Nov 25 18:57:10 2025 ...... Completed iteration: 66 | converge: 0.0009807
Tue Nov 25 18:57:10 2025 .. Calculating final decontaminated matrix
Tue Nov 25 18:57:11 2025 .. Analyzing cells in batch 'GSE155512_2'
Tue Nov 25 18

In [102]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE155512/GSE155512_merged_postR.h5ad")

In [103]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [104]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=4000) # acc to author
# sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 8867


Number of cells before counts filter: 8867
Number of cells beforeMT filter: 8867
Number of cells after MT filter: 8866


In [105]:
adata_final.write("data/public_data/AS_atlas/GSE155512/GSE155512_postQC-R.h5ad")

In [106]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE155512/GSE155512_postQC-R.h5ad")

In [107]:
adata_final.obs['sample'].value_counts()

sample
GSE155512_2    3486
GSE155512_3    2766
GSE155512_1    2614
Name: count, dtype: int64

## GSE159677 51981 44748/45581

In [32]:
file_paths = [
    "data/public_data/AS_atlas/GSE159677/GSM4837523/GSE159677_GSM4837523.h5ad",
    "data/public_data/AS_atlas/GSE159677/GSM4837524/GSE159677_GSM4837524.h5ad",
    "data/public_data/AS_atlas/GSE159677/GSM4837525/GSE159677_GSM4837525.h5ad",
    "data/public_data/AS_atlas/GSE159677/GSM4837526/GSE159677_GSM4837526.h5ad",
    "data/public_data/AS_atlas/GSE159677/GSM4837527/GSE159677_GSM4837527.h5ad",
    "data/public_data/AS_atlas/GSE159677/GSM4837528/GSE159677_GSM4837528.h5ad",
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
]

In [33]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 11015
Number of cells in sample Sample2: 3716
Number of cells in sample Sample3: 15960
Number of cells in sample Sample4: 5523
Number of cells in sample Sample5: 12388
Number of cells in sample Sample6: 3379


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [34]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE159677/GSE159677_merged.h5ad")

In [108]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [109]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE159677/GSE159677_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE159677/GSE159677_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 19:05:53 2025 .. Analyzing cells in batch 'GSE159677_1'
Tue Nov 25 19:05:54 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:06:21 2025 .... Estimating contamination
Tue Nov 25 19:06:26 2025 ...... Completed iteration: 10 | converge: 0.03281
Tue Nov 25 19:06:30 2025 ...... Completed iteration: 20 | converge: 0.01245
Tue Nov 25 19:06:34 2025 ...... Completed iteration: 30 | converge: 0.007319
Tue Nov 25 19:06:39 2025 ...... Completed iteration: 40 | converge: 0.005271
Tue Nov 25 19:06:43 2025 ...... Completed iteration: 50 | converge: 0.004626
Tue Nov 25 19:06:48 2025 ...... Completed iteration: 60 | converge: 0.003932
Tue Nov 25 19:06:53 2025 ...... Completed iteration: 70 | converge: 0.003411
Tue Nov 25 19:06:57 2025 ...... Completed iteration: 80 | converge: 0.003043
Tue Nov 25 19:07:01 2025 ...... Completed iteration: 90 | converge: 0.003

In [117]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE159677/GSE159677_merged_postR.h5ad")

In [118]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [119]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #according to paper
sc.pp.filter_cells(adata_final, max_genes=4000) #according to paper
sc.pp.filter_genes(adata_final, min_cells=3) #according to paper

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
# sc.pp.filter_cells(adata_final, max_counts = 20000) ##according to paper


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #according to paper
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 51981
Number of cells beforeMT filter: 50925
Number of cells after MT filter: 45581


In [120]:
adata_final.write("data/public_data/AS_atlas/GSE159677/GSE159677_postQC-R.h5ad")

In [121]:
adata_final= sc.read_h5ad("data/public_data/AS_atlas/GSE159677/GSE159677_postQC-R.h5ad")

In [122]:
adata_final.obs['sample'].value_counts()

sample
GSE159677_3    14157
GSE159677_5    10462
GSE159677_1    10151
GSE159677_4     4896
GSE159677_2     3101
GSE159677_6     2814
Name: count, dtype: int64

## GSE179159 21810 20262

In [40]:
file_paths = [
    "data/public_data/AS_atlas/GSE179159/GSM5410853/GSE179159_GSM5410853.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSM5410855/GSE179159_GSM5410855.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSM6976304/GSE179159_GSM6976304.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSM6976306/GSE179159_GSM6976306.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSM6976308/GSE179159_GSM6976308.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]

In [41]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 2729
Number of cells in sample Sample2: 1855
Number of cells in sample Sample3: 5961
Number of cells in sample Sample4: 8180
Number of cells in sample Sample5: 3085


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [42]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE179159/GSE179159_merged.h5ad")

In [123]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [124]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE179159/GSE179159_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE179159/GSE179159_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 19:27:04 2025 .. Analyzing cells in batch 'GSE179159_1'
Tue Nov 25 19:27:04 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:27:17 2025 .... Estimating contamination
Tue Nov 25 19:27:18 2025 ...... Completed iteration: 10 | converge: 0.02768
Tue Nov 25 19:27:19 2025 ...... Completed iteration: 20 | converge: 0.0105
Tue Nov 25 19:27:19 2025 ...... Completed iteration: 30 | converge: 0.005044
Tue Nov 25 19:27:20 2025 ...... Completed iteration: 40 | converge: 0.003169
Tue Nov 25 19:27:21 2025 ...... Completed iteration: 50 | converge: 0.002537
Tue Nov 25 19:27:22 2025 ...... Completed iteration: 60 | converge: 0.002032
Tue Nov 25 19:27:22 2025 ...... Completed iteration: 70 | converge: 0.001634
Tue Nov 25 19:27:23 2025 ...... Completed iteration: 80 | converge: 0.001395
Tue Nov 25 19:27:24 2025 ...... Completed iteration: 90 | converge: 0.0011

In [125]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE179159/GSE179159_merged_postR.h5ad")

In [126]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [127]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)
sc.pp.filter_cells(adata_final, max_genes=4000)
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000)
print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 21810
Number of cells before counts filter: 20627
Number of cells beforeMT filter: 20476
Number of cells after MT filter: 20262


In [128]:
adata_final.write("data/public_data/AS_atlas/GSE179159/GSE179159_postQC-R.h5ad")

In [129]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE179159/GSE179159_postQC-R.h5ad")

In [130]:
adata_final.obs['sample'].value_counts()

sample
GSE179159_4    7257
GSE179159_3    5946
GSE179159_5    2777
GSE179159_1    2559
GSE179159_2    1723
Name: count, dtype: int64

## GSE210152 34456 30415

In [11]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE210152/GSE210152_all.h5ad")

In [12]:
adata_final.obs['sample'].value_counts()

sample
GSE210152_6    7450
GSE210152_5    6741
GSE210152_2    5701
GSE210152_4    5279
GSE210152_1    5078
GSE210152_3    4207
Name: count, dtype: int64

In [133]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [134]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE210152/GSE210152_all.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE210152/GSE210152_all_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 19:37:56 2025 .. Analyzing cells in batch 'GSE210152_1'
Tue Nov 25 19:37:57 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:38:17 2025 .... Estimating contamination
Tue Nov 25 19:38:18 2025 ...... Completed iteration: 10 | converge: 0.02657
Tue Nov 25 19:38:19 2025 ...... Completed iteration: 20 | converge: 0.01021
Tue Nov 25 19:38:21 2025 ...... Completed iteration: 30 | converge: 0.006034
Tue Nov 25 19:38:22 2025 ...... Completed iteration: 40 | converge: 0.003914
Tue Nov 25 19:38:23 2025 ...... Completed iteration: 50 | converge: 0.003131
Tue Nov 25 19:38:24 2025 ...... Completed iteration: 60 | converge: 0.002616
Tue Nov 25 19:38:25 2025 ...... Completed iteration: 70 | converge: 0.001894
Tue Nov 25 19:38:26 2025 ...... Completed iteration: 80 | converge: 0.001443
Tue Nov 25 19:38:27 2025 ...... Completed iteration: 90 | converge: 0.001

In [135]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE210152/GSE210152_all_postR.h5ad")

In [136]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [137]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100) 
sc.pp.filter_cells(adata_final, max_genes=5000) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 34456
Number of cells before counts filter: 32717
Number of cells beforeMT filter: 32545
Number of cells after MT filter: 30415


In [138]:
adata_final.write("data/public_data/AS_atlas/GSE210152/GSE210152_postQC-R.h5ad")

In [139]:
adata_final=sc.read_h5ad("data/public_data/AS_atlas/GSE210152/GSE210152_postQC-R.h5ad")

In [140]:
adata_final.obs['sample'].value_counts()

sample
GSE210152_6    6836
GSE210152_5    6035
GSE210152_2    4823
GSE210152_4    4798
GSE210152_1    4515
GSE210152_3    3408
Name: count, dtype: int64

## GSE224273 10441 9931

In [64]:
file_paths = [
    "data/public_data/AS_atlas/GSE224273/GSM7018579_Sample1/GSE224273_GSM7018579.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018580_Sample2/GSE224273_GSM7018580.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018581_Sample3/GSE224273_GSM7018581.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018582_Sample3A/GSE224273_GSM7018582.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018583_Sample4/GSE224273_GSM7018583.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018584_Sample5/GSE224273_GSM7018584.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSM7018585_Sample1G/GSE224273_GSM7018585.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7"
]

In [65]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 289
Number of cells in sample Sample2: 1738
Number of cells in sample Sample3: 814
Number of cells in sample Sample4: 277
Number of cells in sample Sample5: 984
Number of cells in sample Sample6: 1578
Number of cells in sample Sample7: 4761


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [66]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE224273/GSE224273_merged.h5ad")

In [141]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [142]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE224273/GSE224273_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE224273/GSE224273_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 19:46:42 2025 .. Analyzing cells in batch 'GSE224273_1'
Tue Nov 25 19:46:42 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:46:47 2025 .... Estimating contamination
Tue Nov 25 19:46:47 2025 ...... Completed iteration: 10 | converge: 0.01747
Tue Nov 25 19:46:47 2025 ...... Completed iteration: 20 | converge: 0.003732
Tue Nov 25 19:46:47 2025 ...... Completed iteration: 30 | converge: 0.001464
Tue Nov 25 19:46:47 2025 ...... Completed iteration: 34 | converge: 0.0009376
Tue Nov 25 19:46:47 2025 .. Calculating final decontaminated matrix
Tue Nov 25 19:46:47 2025 .. Analyzing cells in batch 'GSE224273_2'
Tue Nov 25 19:46:47 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:46:58 2025 .... Estimating contamination
Tue Nov 25 19:46:58 2025 ...... Completed iteration: 10 | converge: 0.02856
Tue Nov 25 19:46:59 2025 ...... Completed

In [147]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE224273/GSE224273_merged_postR.h5ad")

In [148]:
adata_final

AnnData object with n_obs × n_vars = 10441 × 45091
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE224273_1_UMAP', 'decontX_GSE224273_2_UMAP', 'decontX_GSE224273_3_UMAP', 'decontX_GSE224273_4_UMAP', 'decontX_GSE224273_5_UMAP', 'decontX_GSE224273_6_UMAP', 'decontX_GSE224273_7_UMAP'
    layers: 'decontXcounts'

In [149]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [150]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=4000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 10441
Number of cells before counts filter: 10257
Number of cells beforeMT filter: 10097
Number of cells after MT filter: 9931


In [151]:
adata_final.write("data/public_data/AS_atlas/GSE224273/GSE224273_postQC-R.h5ad")

In [152]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE224273/GSE224273_postQC-R.h5ad")

In [153]:
adata_final.obs['sample'].value_counts()

sample
GSE224273_7    4614
GSE224273_2    1631
GSE224273_6    1546
GSE224273_5     936
GSE224273_3     758
GSE224273_4     237
GSE224273_1     209
Name: count, dtype: int64

## GSE247238 10370 8191/4752

In [93]:
file_paths = [
    "data/public_data/AS_atlas/GSE247238/GSM7885766/GSE247238_GSM7885766.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885767/GSE247238_GSM7885767.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885768/GSE247238_GSM7885768.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885769/GSE247238_GSM7885769.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885770/GSE247238_GSM7885770.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885771/GSE247238_GSM7885771.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885772/GSE247238_GSM7885772.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885773/GSE247238_GSM7885773.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885774/GSE247238_GSM7885774.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885775/GSE247238_GSM7885775.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885776/GSE247238_GSM7885776.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885777/GSE247238_GSM7885777.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885778/GSE247238_GSM7885778.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885779/GSE247238_GSM7885779.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885780/GSE247238_GSM7885780.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885781/GSE247238_GSM7885781.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885782/GSE247238_GSM7885782.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885783/GSE247238_GSM7885783.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSM7885784/GSE247238_GSM7885784.h5ad",
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10",
    "Sample11",
    "Sample12",
    "Sample13",
    "Sample14",
    "Sample15",
    "Sample16",
    "Sample17",
    "Sample18",
    "Sample19"
]

In [94]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 327
Number of cells in sample Sample2: 246
Number of cells in sample Sample3: 296
Number of cells in sample Sample4: 312
Number of cells in sample Sample5: 263
Number of cells in sample Sample6: 270
Number of cells in sample Sample7: 612
Number of cells in sample Sample8: 575
Number of cells in sample Sample9: 631
Number of cells in sample Sample10: 314
Number of cells in sample Sample11: 1832
Number of cells in sample Sample12: 283
Number of cells in sample Sample13: 361
Number of cells in sample Sample14: 186
Number of cells in sample Sample15: 743
Number of cells in sample Sample16: 751
Number of cells in sample Sample17: 648
Number of cells in sample Sample18: 704
Number of cells in sample Sample19: 1016


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [95]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE247238/GSE247238_merged.h5ad")

In [154]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_4010633/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [155]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE247238/GSE247238_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE247238/GSE247238_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 19:55:41 2025 .. Analyzing cells in batch 'GSE247238_1'
Tue Nov 25 19:55:41 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:55:45 2025 .... Estimating contamination
Tue Nov 25 19:55:45 2025 ...... Completed iteration: 10 | converge: 0.01836
Tue Nov 25 19:55:46 2025 ...... Completed iteration: 20 | converge: 0.00386
Tue Nov 25 19:55:46 2025 ...... Completed iteration: 30 | converge: 0.001605
Tue Nov 25 19:55:46 2025 ...... Completed iteration: 38 | converge: 0.0009087
Tue Nov 25 19:55:46 2025 .. Calculating final decontaminated matrix
Tue Nov 25 19:55:46 2025 .. Analyzing cells in batch 'GSE247238_2'
Tue Nov 25 19:55:46 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 19:55:50 2025 .... Estimating contamination
Tue Nov 25 19:55:50 2025 ...... Completed iteration: 10 | converge: 0.0129
Tue Nov 25 19:55:51 2025 ...... Completed i

In [156]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE247238/GSE247238_merged_postR.h5ad")

In [157]:
adata_final

AnnData object with n_obs × n_vars = 10370 × 60663
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE247238_10_UMAP', 'decontX_GSE247238_11_UMAP', 'decontX_GSE247238_12_UMAP', 'decontX_GSE247238_13_UMAP', 'decontX_GSE247238_14_UMAP', 'decontX_GSE247238_15_UMAP', 'decontX_GSE247238_16_UMAP', 'decontX_GSE247238_17_UMAP', 'decontX_GSE247238_18_UMAP', 'decontX_GSE247238_19_UMAP', 'decontX_GSE247238_1_UMAP', 'decontX_GSE247238_2_UMAP', 'decontX_GSE247238_3_UMAP', 'decontX_GSE247238_4_UMAP', 'decontX_GSE247238_5_UMAP', 'decontX_GSE247238_6_UMAP', 'decontX_GSE247238_7_UMAP', 'decontX_GSE247238_8_UMAP', 'decontX_GSE247238_9_UMAP'
    layers: 'decontXcounts'

In [158]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [159]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=10000)
# sc.pp.filter_genes(adata_final, min_cells=1) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, min_counts = 1000) 
sc.pp.filter_cells(adata_final, max_counts = 50000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 10370
Number of cells before counts filter: 7673
Number of cells beforeMT filter: 5079
Number of cells after MT filter: 4752


In [160]:
adata_final.write("data/public_data/AS_atlas/GSE247238/GSE247238_postQC-R.h5ad")

In [161]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE247238/GSE247238_postQC-R.h5ad")

In [162]:
adata_final.obs['sample'].value_counts()

sample
GSE247238_11    1321
GSE247238_19     846
GSE247238_9      315
GSE247238_16     310
GSE247238_15     297
GSE247238_8      297
GSE247238_7      281
GSE247238_12     164
GSE247238_10     157
GSE247238_13     119
GSE247238_1      114
GSE247238_4      102
GSE247238_3       87
GSE247238_6       86
GSE247238_5       81
GSE247238_2       76
GSE247238_14      38
GSE247238_17      36
GSE247238_18      25
Name: count, dtype: int64

## GSE234077 83121 72223/66946

In [3]:
file_paths = [
    "data/public_data/AS_atlas/GSE234077/GSM7445263/GSE234077_GSM7445263.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445264/GSE234077_GSM7445264.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445265/GSE234077_GSM7445265.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445266/GSE234077_GSM7445266.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445267/GSE234077_GSM7445267.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445268/GSE234077_GSM7445268.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445269/GSE234077_GSM7445269.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445270/GSE234077_GSM7445270.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445271/GSE234077_GSM7445271.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSM7445272/GSE234077_GSM7445272.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10"
]

In [4]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 2444
Number of cells in sample Sample2: 12394
Number of cells in sample Sample3: 4665
Number of cells in sample Sample4: 1734
Number of cells in sample Sample5: 9770
Number of cells in sample Sample6: 8422
Number of cells in sample Sample7: 10379
Number of cells in sample Sample8: 11300
Number of cells in sample Sample9: 15358
Number of cells in sample Sample10: 6655


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE234077/GSE234077_merged.h5ad")

In [ ]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [164]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE234077/GSE234077_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE234077/GSE234077_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 20:14:17 2025 .. Analyzing cells in batch 'GSE234077_1'
Tue Nov 25 20:14:17 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 20:14:29 2025 .... Estimating contamination
Tue Nov 25 20:14:30 2025 ...... Completed iteration: 10 | converge: 0.02558
Tue Nov 25 20:14:31 2025 ...... Completed iteration: 20 | converge: 0.009533
Tue Nov 25 20:14:32 2025 ...... Completed iteration: 30 | converge: 0.005404
Tue Nov 25 20:14:32 2025 ...... Completed iteration: 40 | converge: 0.003342
Tue Nov 25 20:14:33 2025 ...... Completed iteration: 50 | converge: 0.002374
Tue Nov 25 20:14:34 2025 ...... Completed iteration: 60 | converge: 0.001757
Tue Nov 25 20:14:35 2025 ...... Completed iteration: 70 | converge: 0.001137
Tue Nov 25 20:14:35 2025 ...... Completed iteration: 76 | converge: 0.0009994
Tue Nov 25 20:14:35 2025 .. Calculating final decontaminated matrix
Tue

In [165]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE234077/GSE234077_merged_postR.h5ad")

In [166]:
adata_final

AnnData object with n_obs × n_vars = 83121 × 36601
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE234077_10_UMAP', 'decontX_GSE234077_1_UMAP', 'decontX_GSE234077_2_UMAP', 'decontX_GSE234077_3_UMAP', 'decontX_GSE234077_4_UMAP', 'decontX_GSE234077_5_UMAP', 'decontX_GSE234077_6_UMAP', 'decontX_GSE234077_7_UMAP', 'decontX_GSE234077_8_UMAP', 'decontX_GSE234077_9_UMAP'
    layers: 'decontXcounts'

In [167]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [168]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  # author
sc.pp.filter_cells(adata_final, max_genes=10000) # author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 200)
sc.pp.filter_cells(adata_final, max_counts = 10000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 83121
Number of cells before counts filter: 81854
Number of cells beforeMT filter: 73547
Number of cells after MT filter: 66946


In [169]:
adata_final.write("data/public_data/AS_atlas/GSE234077/GSE234077_postQC-R.h5ad")

In [170]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE234077/GSE234077_postQC-R.h5ad")

In [171]:
adata_final.obs['sample'].value_counts()

sample
GSE234077_9     13004
GSE234077_8      9921
GSE234077_2      9207
GSE234077_7      8904
GSE234077_5      8861
GSE234077_6      6863
GSE234077_10     4904
GSE234077_3      2653
GSE234077_1      2037
GSE234077_4       592
Name: count, dtype: int64

## GSE253904 81942 67835/77112

In [12]:
file_paths = [
    "data/public_data/AS_atlas/GSE253904/GSM8029950/GSE253904_GSM8029950.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029952/GSE253904_GSM8029952.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029954/GSE253904_GSM8029954.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029956/GSE253904_GSM8029956.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029958/GSE253904_GSM8029958.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029960/GSE253904_GSM8029960.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029962/GSE253904_GSM8029962.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029963/GSE253904_GSM8029963.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029964/GSE253904_GSM8029964.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029965/GSE253904_GSM8029965.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029966/GSE253904_GSM8029966.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029967/GSE253904_GSM8029967.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029968/GSE253904_GSM8029968.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029969/GSE253904_GSM8029969.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029970/GSE253904_GSM8029970.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029971/GSE253904_GSM8029971.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029972/GSE253904_GSM8029972.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSM8029973/GSE253904_GSM8029973.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10",
    "Sample11",
    "Sample12",
    "Sample13",
    "Sample14",
    "Sample15",
    "Sample16",
    "Sample17",
    "Sample18",
    "Sample19"
]

In [13]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 2143
Number of cells in sample Sample2: 4673
Number of cells in sample Sample3: 5912
Number of cells in sample Sample4: 4368
Number of cells in sample Sample5: 4473
Number of cells in sample Sample6: 4657
Number of cells in sample Sample7: 2715
Number of cells in sample Sample8: 6691
Number of cells in sample Sample9: 5432
Number of cells in sample Sample10: 3543
Number of cells in sample Sample11: 4335
Number of cells in sample Sample12: 5147
Number of cells in sample Sample13: 5648
Number of cells in sample Sample14: 6299
Number of cells in sample Sample15: 4843
Number of cells in sample Sample16: 4937
Number of cells in sample Sample17: 2316
Number of cells in sample Sample18: 3810


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [14]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE253904/GSE253904_merged.h5ad")

In [ ]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [172]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE253904/GSE253904_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE253904/GSE253904_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 20:36:11 2025 .. Analyzing cells in batch 'GSE253904_1'
Tue Nov 25 20:36:11 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 20:36:21 2025 .... Estimating contamination
Tue Nov 25 20:36:23 2025 ...... Completed iteration: 10 | converge: 0.02342
Tue Nov 25 20:36:24 2025 ...... Completed iteration: 20 | converge: 0.008797
Tue Nov 25 20:36:25 2025 ...... Completed iteration: 30 | converge: 0.004711
Tue Nov 25 20:36:26 2025 ...... Completed iteration: 40 | converge: 0.002926
Tue Nov 25 20:36:27 2025 ...... Completed iteration: 50 | converge: 0.001967
Tue Nov 25 20:36:28 2025 ...... Completed iteration: 60 | converge: 0.001288
Tue Nov 25 20:36:29 2025 ...... Completed iteration: 68 | converge: 0.0009782
Tue Nov 25 20:36:29 2025 .. Calculating final decontaminated matrix
Tue Nov 25 20:36:30 2025 .. Analyzing cells in batch 'GSE253904_2'
Tue Nov 25 20

In [3]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE253904/GSE253904_merged_postR.h5ad")

In [4]:
adata_final

AnnData object with n_obs × n_vars = 81942 × 36601
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE253904_10_UMAP', 'decontX_GSE253904_11_UMAP', 'decontX_GSE253904_12_UMAP', 'decontX_GSE253904_13_UMAP', 'decontX_GSE253904_14_UMAP', 'decontX_GSE253904_15_UMAP', 'decontX_GSE253904_16_UMAP', 'decontX_GSE253904_17_UMAP', 'decontX_GSE253904_18_UMAP', 'decontX_GSE253904_1_UMAP', 'decontX_GSE253904_2_UMAP', 'decontX_GSE253904_3_UMAP', 'decontX_GSE253904_4_UMAP', 'decontX_GSE253904_5_UMAP', 'decontX_GSE253904_6_UMAP', 'decontX_GSE253904_7_UMAP', 'decontX_GSE253904_8_UMAP', 'decontX_GSE253904_9_UMAP'
    layers: 'decontXcounts'

In [6]:
# adata_final.var

""
MIR1302-2HG
FAM138A
OR4F5
AL627309.1
AL627309.3
...
AC141272.1
AC023491.2
AC007325.1
AC007325.4


In [7]:
# # adata_bashore.obs["sample"] = [sample + "_Bashore" for sample in adata_bashore.obs["sample"]]
# #map gene ids to ensembl
# ensembl_id_df = pd.read_csv("/home/lixiangyu/zr/Annotate/gene_names_to_ensembl_ALLFOUND_allfernandez_no6_withallslysz.csv")
# gene_to_ensembl = dict(zip(ensembl_id_df['gene_name'], ensembl_id_df['ensembl_id']))
# # Map the variable names in AnnData
# adata_final.var['ensembl_id'] = [gene_to_ensembl[gene] if gene in gene_to_ensembl else gene for gene in adata_final.var_names]

In [8]:
# adata_final.var

,ensembl_id
MIR1302-2HG,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092
AL627309.1,ENSG00000238009
AL627309.3,ENSG00000239945
...,...
AC141272.1,ENSG00000277836
AC023491.2,ENSG00000278633
AC007325.1,ENSG00000276017
AC007325.4,ENSG00000278817


In [9]:
# adata_final.write_h5ad("data/public_data/AS_atlas/GSE253904/GSE253904_merged_FM.h5ad")

In [5]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [6]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=6000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 40000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 30] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 81942
Number of cells before counts filter: 80265
Number of cells beforeMT filter: 79837
Number of cells after MT filter: 77112


In [7]:
adata_final.write("data/public_data/AS_atlas/GSE253904/GSE253904_postQC-R.h5ad")

In [8]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE253904/GSE253904_postQC-R.h5ad")

In [9]:
adata_final.obs['sample'].value_counts()

sample
GSE253904_8     6262
GSE253904_14    6074
GSE253904_3     5706
GSE253904_13    5511
GSE253904_9     5205
GSE253904_12    5040
GSE253904_16    4705
GSE253904_15    4458
GSE253904_2     4287
GSE253904_5     4252
GSE253904_4     4107
GSE253904_6     4054
GSE253904_11    3790
GSE253904_18    3473
GSE253904_10    3432
GSE253904_7     2590
GSE253904_17    2188
GSE253904_1     1978
Name: count, dtype: int64

## GSE131778 11756 11501

In [22]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE131778/GSE131778.h5ad")

In [23]:
adata_final.obs['sample'].value_counts()

sample
GSE131778_6    2385
GSE131778_7    2089
GSE131778_5    1847
GSE131778_8    1752
GSE131778_3    1491
GSE131778_4    1400
GSE131778_1     530
GSE131778_2     262
Name: count, dtype: int64

In [32]:
adata_final

AnnData object with n_obs × n_vars = 11756 × 20431
    obs: 'sample', 'dataset', 'symptoms', 'intervention', 'location', 'gender', 'age'

In [11]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_172592/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [12]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE131778/GSE131778.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE131778/GSE131778_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [13]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE131778/GSE131778_postR.h5ad")

In [14]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [15]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100) 
sc.pp.filter_cells(adata_final, max_genes=4000) 
sc.pp.filter_genes(adata_final, min_cells=3) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 7.5]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 11756
Number of cells before counts filter: 11756
Number of cells beforeMT filter: 11501
Number of cells after MT filter: 11501


In [16]:
adata_final.write("data/public_data/AS_atlas/GSE131778/GSE131778_postQC-R.h5ad")

In [17]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE131778/GSE131778_postQC-R.h5ad")

In [18]:
adata_final.obs['sample'].value_counts()

sample
GSE131778_6    2367
GSE131778_7    2065
GSE131778_5    1814
GSE131778_8    1738
GSE131778_3    1491
GSE131778_4    1400
GSE131778_1     416
GSE131778_2     210
Name: count, dtype: int64

## GSE184073 2791 2371/1693

In [37]:
file_paths = [
    "data/public_data/AS_atlas/GSE184073/GSM5577199/GSE184073_GSM5577199.h5ad",
    "data/public_data/AS_atlas/GSE184073/GSM5577200/GSE184073_GSM5577200.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]

In [38]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 1071
Number of cells in sample Sample2: 1720


In [39]:
adata_final.write_h5ad("data/public_data/AS_atlas/GSE184073/GSE184073_merged.h5ad")

In [ ]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [19]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE184073/GSE184073_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE184073/GSE184073_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 21:10:54 2025 .. Analyzing cells in batch 'GSE184073_1'
Tue Nov 25 21:10:54 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 21:11:01 2025 .... Estimating contamination
Tue Nov 25 21:11:01 2025 ...... Completed iteration: 10 | converge: 0.02648
Tue Nov 25 21:11:02 2025 ...... Completed iteration: 20 | converge: 0.008946
Tue Nov 25 21:11:02 2025 ...... Completed iteration: 30 | converge: 0.00521
Tue Nov 25 21:11:03 2025 ...... Completed iteration: 40 | converge: 0.003153
Tue Nov 25 21:11:03 2025 ...... Completed iteration: 50 | converge: 0.001998
Tue Nov 25 21:11:04 2025 ...... Completed iteration: 60 | converge: 0.001551
Tue Nov 25 21:11:04 2025 ...... Completed iteration: 70 | converge: 0.001375
Tue Nov 25 21:11:05 2025 ...... Completed iteration: 77 | converge: 0.0009675
Tue Nov 25 21:11:05 2025 .. Calculating final decontaminated matrix
Tue 

In [20]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE184073/GSE184073_merged_postR.h5ad")

In [21]:
adata_final

AnnData object with n_obs × n_vars = 2791 × 36601
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE184073_1_UMAP', 'decontX_GSE184073_2_UMAP'
    layers: 'decontXcounts'

In [22]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [23]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

# sc.pp.filter_cells(adata_final, max_counts = 50000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 8] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 2791


Number of cells beforeMT filter: 2327
Number of cells after MT filter: 1693


In [24]:
adata_final.write("data/public_data/AS_atlas/GSE184073/GSE184073_postQC-R.h5ad")

In [25]:
adata_final = sc.read_h5ad("data/public_data/AS_atlas/GSE184073/GSE184073_postQC-R.h5ad")

In [26]:
adata_final.obs['sample'].value_counts()

sample
GSE184073_2    1027
GSE184073_1     666
Name: count, dtype: int64

## GSE196943 26352 26330

In [4]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE196943/GSE196943.h5ad")

In [5]:
adata_final.obs['sample'].value_counts()

sample
GSE196943_2     5233
GSE196943_6     4462
GSE196943_1     3912
GSE196943_5     3632
GSE196943_12    1956
GSE196943_10    1871
GSE196943_4     1564
GSE196943_11    1548
GSE196943_3     1121
GSE196943_8      522
GSE196943_9      386
GSE196943_7      145
Name: count, dtype: int64

In [6]:
adata_final

AnnData object with n_obs × n_vars = 26352 × 23858
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [ ]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [27]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/public_data/AS_atlas/GSE196943/GSE196943.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/public_data/AS_atlas/GSE196943/GSE196943_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Tue Nov 25 21:17:35 2025 .. Analyzing cells in batch 'GSE196943_3'
Tue Nov 25 21:17:35 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 21:17:44 2025 .... Estimating contamination
Tue Nov 25 21:17:45 2025 ...... Completed iteration: 10 | converge: 0.01476
Tue Nov 25 21:17:45 2025 ...... Completed iteration: 20 | converge: 0.005187
Tue Nov 25 21:17:45 2025 ...... Completed iteration: 30 | converge: 0.002225
Tue Nov 25 21:17:46 2025 ...... Completed iteration: 40 | converge: 0.001583
Tue Nov 25 21:17:46 2025 ...... Completed iteration: 50 | converge: 0.001172
Tue Nov 25 21:17:46 2025 ...... Completed iteration: 60 | converge: 0.0009918
Tue Nov 25 21:17:46 2025 .. Calculating final decontaminated matrix
Tue Nov 25 21:17:46 2025 .. Analyzing cells in batch 'GSE196943_6'
Tue Nov 25 21:17:47 2025 .... Generating UMAP and estimating cell types
Tue Nov 25 21:18:1

In [28]:
adata_final =sc.read_h5ad("data/public_data/AS_atlas/GSE196943/GSE196943_postR.h5ad")

In [29]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [30]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=4000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 26352
Number of cells before counts filter: 26330
Number of cells beforeMT filter: 26330
Number of cells after MT filter: 26330


In [31]:
adata_final

View of AnnData object with n_obs × n_vars = 26330 × 20652
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE196943_10_UMAP', 'decontX_GSE196943_11_UMAP', 'decontX_GSE196943_12_UMAP', 'decontX_GSE196943_1_UMAP', 'decontX_GSE196943_2_UMAP', 'decontX_GSE196943_3_UMAP', 'decontX_GSE196943_4_UMAP', 'decontX_GSE196943_5_UMAP', 'decontX_GSE196943_6_UMAP', 'decontX_GSE196943_7_UMAP', 'decontX_GSE196943_8_UMAP', 'decontX_GSE196943_9_UMAP'
    layers: 'decontXcounts'

In [32]:
adata_final.write("data/public_data/AS_atlas/GSE196943/GSE196943_postQC-R.h5ad")

In [33]:
adata_final.obs['sample'].value_counts()

sample
GSE196943_2     5233
GSE196943_6     4458
GSE196943_1     3912
GSE196943_5     3629
GSE196943_12    1955
GSE196943_10    1864
GSE196943_4     1564
GSE196943_11    1542
GSE196943_3     1121
GSE196943_8      522
GSE196943_9      386
GSE196943_7      144
Name: count, dtype: int64

# Healthy(3)

## adata_1-normalized

In [43]:
adata_1 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/cellbygene_vascular_tissue-premerge.h5ad")

In [44]:
adata_1

AnnData object with n_obs × n_vars = 113304 × 33159
    obs: 'mapped_reference_annotation', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_derivation_process', 'donor_BMI_at_collection', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_uuid', 'suspension_type', 'library_uuid', 'assay_ontology_term_id', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observation_joinid', 'dataset', 'sample', 'intervention'
    var: 'feature_is_filtered', 'gene_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 

In [45]:
# 查看数据的基本统计量
print("数据最小值:", adata_1.X.min())
print("数据最大值:", adata_1.X.max())
print("数据均值:", adata_1.X.mean())

数据最小值: 0.0
数据最大值: 8.95153
数据均值: 0.09474795


## adata_2 39480 34612

In [3]:
##分开的样本
adata_2 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/cellbygene_TS_Vasculature-premerge.h5ad")

In [4]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observation_j

In [5]:
adata_2.obs['sample'].value_counts()

sample
TSP25_Vasculature_Aorta_10X_1_1                                     6018
TSP14_Vasculature_CoronaryArteries_10X_1_1                          5878
TSP21_Vasculature_Aorta_10X_1_1                                     5237
TSP25_Vasculature_CoronaryArteries_10X_1_1                          5124
TSP2_Vasculature_Aorta_10X_1_1                                      3898
TSP2_Vasculature_Aorta_10X_1_2                                      3757
TSP25_Vasculature_AortaThoracic_10X_2_1                             2586
TSP21_Vasculature_Aorta_10X_2_1                                     1644
TSP21_Vasculature_CoronaryArteries_10X_1_1                          1443
TSP21_Vasculature_CoronaryArteries_10X_2_1                          1435
TSP2_Vasculature_Aorta_10X_2_2                                       411
TSP2_Vasculature_Aorta_10X_2_1                                       405
TSP2_Vasculature_Aorta_SS2_B113343_B133091_Immune                    293
TSP2_Vasculature_Aorta_SS2_B114577_B133059_E

In [6]:
# 将raw中的数据恢复到adata.X（同时更新var为raw的var）
adata_2 = adata_2.raw.to_adata()

In [7]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ensg,,,,,,,,,,,,,,,
ENSG00000000003,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,TNMD,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,DPM1,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [8]:
# 查看数据的基本统计量
print("数据最小值:", adata_2.X.min())
print("数据最大值:", adata_2.X.max())
print("数据均值:", adata_2.X.mean())

数据最小值: 0.0
数据最大值: 1245867.0
数据均值: 1.057604


In [9]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observation_j

In [10]:
adata_2.var_names = adata_2.var['feature_name']

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:947: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "var")


In [11]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_name,feature_reference,feature_biotype,feature_length,feature_type
feature_name,,,,,,,,,,,,,,,
TSPAN6,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
TNMD,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,TNMD,NCBITaxon:9606,gene,873,protein_coding
DPM1,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,DPM1,NCBITaxon:9606,gene,1262,protein_coding
SCYL3,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
C1orf112,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [23]:
# # adata_2.X = adata_2.layers["decontXcounts"]
# # 备份原始X数据到layers中
# adata_2.layers["scales_counts"] = adata_2.X.copy()
# # 替换X为去污染后的数据
# # adata_2.X = adata_2.layers["scale_data"]
# adata_2.X = adata_2.layers["decontXcounts"]

In [24]:
# # 查看数据的基本统计量
# print("数据最小值:", adata_2.X.min())
# print("数据最大值:", adata_2.X.max())
# print("数据均值:", adata_2.X.mean())

数据最小值: 0
数据最大值: 1245867
数据均值: 1.044593453377366


In [12]:
adata_2.obs = adata_2.obs[['dataset','sample','symptoms','gender','age','intervention','tissue']].copy() # 添加copy()确保数据独立
adata_2.obsm = {}
adata_2.layers = {}
adata_2.uns = {}
adata_2.varm = {}
adata_2.obsp = {}
adata_2.var = adata_2.var[['mt','ensembl_id']].copy()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:850: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)


In [13]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue'
    var: 'mt', 'ensembl_id'

In [14]:
adata_2.var

,mt,ensembl_id
feature_name,,
TSPAN6,False,ENSG00000000003.15
TNMD,False,ENSG00000000005.6
DPM1,False,ENSG00000000419.14
SCYL3,False,ENSG00000000457.14
C1orf112,False,ENSG00000000460.17
...,...,...
ENSG00000290162,False,ENSG00000290162.1
ENSG00000290163,False,ENSG00000290163.1
ENSG00000290164,False,ENSG00000290164.1


In [15]:
adata_2.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/cellbygene_TS_Vasculature-prepostQC-1.h5ad")##分开的样本

## adata_2 Tabula Sapiens-Vasculature

In [16]:
adata_2 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/cellbygene_TS_Vasculature-premerge-sample_name.h5ad")##统一的样本

In [17]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observatio

In [18]:
adata_2.obs['sample'].value_counts()

sample
Tabula Sapiens-Vasculature    39480
Name: count, dtype: int64

In [19]:
adata_2 = adata_2.raw.to_adata()

In [20]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ensg,,,,,,,,,,,,,,,
ENSG00000000003,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,TNMD,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,DPM1,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [21]:
# 查看数据的基本统计量
print("数据最小值:", adata_2.X.min())
print("数据最大值:", adata_2.X.max())
print("数据均值:", adata_2.X.mean())

数据最小值: 0.0
数据最大值: 1245867.0
数据均值: 1.057604


In [22]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observatio

In [23]:
adata_2.var_names = adata_2.var['feature_name']

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:947: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "var")


In [24]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_name,feature_reference,feature_biotype,feature_length,feature_type
feature_name,,,,,,,,,,,,,,,
TSPAN6,ENSG00000000003.15,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
TNMD,ENSG00000000005.6,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,TNMD,NCBITaxon:9606,gene,873,protein_coding
DPM1,ENSG00000000419.14,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,DPM1,NCBITaxon:9606,gene,1262,protein_coding
SCYL3,ENSG00000000457.14,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
C1orf112,ENSG00000000460.17,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162.1,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163.1,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164.1,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [25]:
adata_2.obs = adata_2.obs[['dataset','sample','symptoms','gender','age','intervention','tissue']].copy()
adata_2.obsm = {}
adata_2.layers = {}
adata_2.uns = {}
adata_2.varm = {}
adata_2.obsp = {}
adata_2.var = adata_2.var[['mt','ensembl_id']].copy()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:850: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)


In [26]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue'
    var: 'mt', 'ensembl_id'

In [27]:
adata_2.var

,mt,ensembl_id
feature_name,,
TSPAN6,False,ENSG00000000003.15
TNMD,False,ENSG00000000005.6
DPM1,False,ENSG00000000419.14
SCYL3,False,ENSG00000000457.14
C1orf112,False,ENSG00000000460.17
...,...,...
ENSG00000290162,False,ENSG00000290162.1
ENSG00000290163,False,ENSG00000290163.1
ENSG00000290164,False,ENSG00000290164.1


In [28]:
adata_2.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/cellbygene_TS_Vasculature-prepostQC-unify.h5ad")

## adata_2 decountsX

In [42]:
adata_2 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/cellbygene_TS_Vasculature-premerge-sample_name-decontXcounts.h5ad")

In [43]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observatio

In [44]:
adata_2.obs['sample'].value_counts()

sample
Tabula Sapiens-Vasculature    39480
Name: count, dtype: int64

In [45]:
# adata_2 = adata_2.raw.to_adata()

In [46]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_is_filtered,gene_name,feature_reference,feature_biotype,feature_length,feature_type
feature_name,,,,,,,,,,,,,,,,
TSPAN6,ENSG00000000003,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,False,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
TNMD,ENSG00000000005,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,False,TNMD,NCBITaxon:9606,gene,873,protein_coding
DPM1,ENSG00000000419,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,False,DPM1,NCBITaxon:9606,gene,1262,protein_coding
SCYL3,ENSG00000000457,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,False,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
C1orf112,ENSG00000000460,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,False,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,False,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,False,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,False,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [47]:
# 查看数据的基本统计量
print("数据最小值:", adata_2.X.min())
print("数据最大值:", adata_2.X.max())
print("数据均值:", adata_2.X.mean())

数据最小值: 0
数据最大值: 1245867
数据均值: 1.044593453377366


In [48]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'donor_id', 'tissue_in_publication', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'assay_ontology_term_id', 'sample_id', 'replicate', '10X_run', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_type_ontology_term_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'ethnicity_original', 'sample_number', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'cell_type_level1_healthy', 'assay', 'symptoms', 'gender', 'tissue', 'self_reported_ethnicity', 'age', 'observatio

In [49]:
adata_2.var

,ensembl_id,genome,mt,ercc,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,mean,std,feature_is_filtered,gene_name,feature_reference,feature_biotype,feature_length,feature_type
feature_name,,,,,,,,,,,,,,,,
TSPAN6,ENSG00000000003,Gencode_v41,False,False,161872,2.379617,91.800375,4697694.0,0.075550,0.250910,False,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
TNMD,ENSG00000000005,Gencode_v41,False,False,9323,0.220273,99.527743,434850.0,0.005206,0.074072,False,TNMD,NCBITaxon:9606,gene,873,protein_coding
DPM1,ENSG00000000419,Gencode_v41,False,False,461590,3.523875,76.618161,6956619.0,0.262252,0.435713,False,DPM1,NCBITaxon:9606,gene,1262,protein_coding
SCYL3,ENSG00000000457,Gencode_v41,False,False,156149,0.493041,92.090273,973332.0,0.077343,0.256702,False,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
C1orf112,ENSG00000000460,Gencode_v41,False,False,120250,0.281519,93.908737,555757.0,0.066071,0.256870,False,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162,ENSG00000290162,Gencode_v41,False,False,600,0.001394,99.969607,2751.0,0.000201,0.012240,False,ENSG00000290162,NCBITaxon:9606,gene,149,unprocessed_pseudogene
ENSG00000290163,ENSG00000290163,Gencode_v41,False,False,1814,0.006923,99.908112,13667.0,0.000767,0.024918,False,ENSG00000290163,NCBITaxon:9606,gene,144,unprocessed_pseudogene
ENSG00000290164,ENSG00000290164,Gencode_v41,False,False,934,0.005787,99.952688,11424.0,0.000362,0.016829,False,ENSG00000290164,NCBITaxon:9606,gene,138,unprocessed_pseudogene


In [50]:
adata_2.obs = adata_2.obs[['dataset','sample','symptoms','gender','age','intervention','tissue']].copy()
adata_2.obsm = {}
adata_2.layers = {}
adata_2.uns = {}
adata_2.varm = {}
adata_2.obsp = {}
adata_2.var = adata_2.var[['mt','ensembl_id']].copy()

In [51]:
adata_2

AnnData object with n_obs × n_vars = 39480 × 61759
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue'
    var: 'mt', 'ensembl_id'

In [52]:
adata_2.var

,mt,ensembl_id
feature_name,,
TSPAN6,False,ENSG00000000003
TNMD,False,ENSG00000000005
DPM1,False,ENSG00000000419
SCYL3,False,ENSG00000000457
C1orf112,False,ENSG00000000460
...,...,...
ENSG00000290162,False,ENSG00000290162
ENSG00000290163,False,ENSG00000290163
ENSG00000290164,False,ENSG00000290164


In [53]:
adata_2.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/cellbygene_TS_Vasculature-prepostQC-unify-decontXcounts.h5ad")

## adata_3 62398 

In [16]:
##把id加在了var列
adata_3 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/Article_SC_healthy_ensembl_id-premerge.h5ad")

In [17]:
adata_3

AnnData object with n_obs × n_vars = 62398 × 7998
    obs: 'nCount_RNA', 'nFeature_RNA', 'tissue', 'Disease', 'sample', 'Study', 'symptoms', 'Artery', 'Assay', 'percent.mt', 'Doublets', 'gender', '_scvi_batch', '_scvi_labels', 'leiden01', 'leiden02', 'cell_type_level1_healthy', 'cell_type_level1_healthy_Subclustering', 'dataset', 'age', 'intervention'
    var: 'features', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'ensembl_id'
    uns: 'Main_Cell_Types_colors', 'Study_colors', 'Subclustering_colors', '_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'leiden', 'leiden01_colors', 'leiden02_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_scVI', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [18]:
adata_3.X = adata_3.layers["counts"]

In [19]:
adata_3.obs['cell_type_level1_healthy'].value_counts()

cell_type_level1_healthy
VSMC                     18976
Fibroblast               18564
Macrophage               10306
T cells                   7092
Endothelial               3774
Pericytes                 1901
Endothelial (DKK2+)        646
Mast cells                 486
Lymphatic Endothelial      262
Neuronal cells             248
cDC1                        50
Plasma cells                47
B cells                     33
pDC                         12
Neutrophils                  1
Name: count, dtype: int64

In [20]:
# 查看数据的基本统计量
print("数据最小值:", adata_3.X.min())
print("数据最大值:", adata_3.X.max())
print("数据均值:", adata_3.X.mean())

数据最小值: 0.0
数据最大值: 26057.0
数据均值: 0.33994174


In [21]:
adata_3.var

,features,highly_variable,highly_variable_rank,means,variances,variances_norm,highly_variable_nbatches,mean,std,ensembl_id
AL645608.1,AL645608.1,True,4413.5,0.004303,0.004580,1.013226,8,0.006641,0.108423,ENSG00000224969
SAMD11,SAMD11,True,1929.0,0.030008,0.046126,1.293411,13,0.038319,0.257912,ENSG00000187634
AL645608.8,AL645608.8,True,4249.0,0.006710,0.007716,0.923715,7,0.009506,0.128244,ENSG00000273443
HES4,HES4,True,977.0,0.338801,0.958337,1.633104,13,0.333174,0.769423,ENSG00000188290
ISG15,ISG15,True,237.0,0.537354,4.774249,3.860124,13,0.456642,0.906232,ENSG00000187608
...,...,...,...,...,...,...,...,...,...,...
MT-CYB,MT-CYB,True,2625.0,18.579292,324.770542,0.627513,1,4.063208,1.078016,ENSG00000198727
AC141272.1,AC141272.1,True,898.0,0.000059,0.000133,0.161658,1,0.000024,0.005646,ENSG00000277836
AC004556.1,AC004556.1,True,4607.0,0.008376,0.008869,0.650900,1,0.011022,0.129004,ENSG00000276345
AC233755.2,AC233755.2,True,285.0,0.008065,1.922967,1.000565,3,0.000774,0.043835,ENSG00000277856


In [27]:
adata_3.obs['gender'].value_counts()

gender
male      46198
female    16200
Name: count, dtype: int64

In [30]:
adata_3.obs['gender'] = adata_3.obs['gender'].replace('male', 'Male')
adata_3.obs['gender'] = adata_3.obs['gender'].replace('female', 'Female')

/tmp/ipykernel_2343877/3029861963.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata_3.obs['gender'] = adata_3.obs['gender'].replace('female', 'Female')


In [31]:
adata_3.obs['gender'].value_counts()

gender
Male      46198
Female    16200
Name: count, dtype: int64

In [32]:
adata_3.obs = adata_3.obs[['dataset','sample','symptoms','gender','age','intervention','tissue']].copy() # 添加copy()确保数据独立
adata_3.var = adata_3.var[['ensembl_id']].copy()
adata_3.uns = {}
adata_3.obsm = {}
adata_3.varm = {}
adata_3.layers = {}
adata_3.obsp = {}

In [33]:
##把id加在了var列
adata_3.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/Article_SC_healthy_ensembl_id-prepostQC.h5ad")

In [34]:
adata_3

AnnData object with n_obs × n_vars = 62398 × 7998
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue'
    var: 'ensembl_id'

In [175]:
adata_3.obs['sample'].value_counts()

sample
S2    26812
S1    19386
S3    16200
Name: count, dtype: int64

In [176]:
adata_3.var['mt'] = adata_3.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_3, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [177]:
stats = adata_3.obs['pct_counts_mt'].describe()
print(stats)

count    62398.000000
mean         5.191480
std          4.199013
min          0.000000
25%          2.293885
50%          3.896861
75%          6.748904
max         34.441086
Name: pct_counts_mt, dtype: float64


In [178]:
sc.pp.filter_cells(adata_3, min_genes=0) # to get n_genes

In [179]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_3.n_obs))
sc.pp.filter_cells(adata_3, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_3, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_3, min_cells=2) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_3.n_obs))

sc.pp.filter_cells(adata_3, max_counts = 40000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_3.n_obs))

adata_3 = adata_3[adata_3.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_3.n_obs))


Number of cells before gene filter: 62398
Number of cells before counts filter: 62360
Number of cells beforeMT filter: 62360
Number of cells after MT filter: 54850


In [180]:
stats = adata_3.obs['pct_counts_mt'].describe()
print(stats)

count    54850.000000
mean         3.949923
std          2.332908
min          0.000000
25%          2.110536
50%          3.466956
75%          5.430325
max          9.996541
Name: pct_counts_mt, dtype: float64


In [181]:
adata_3

View of AnnData object with n_obs × n_vars = 54850 × 7823
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'ensembl_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'

In [182]:
adata_3.obs['sample'].value_counts()

sample
S2    25206
S3    14891
S1    14753
Name: count, dtype: int64

In [191]:
adata = adata_3[adata_3.obs['sample']=='S3']

In [192]:
adata.obs['gender'].value_counts()

gender
female    14891
Name: count, dtype: int64

In [193]:
adata.obs['tissue'].value_counts()

tissue
Coronary artery    14891
Name: count, dtype: int64

In [183]:
adata_3.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/Article_SC_healthy_ensembl_id-postQC.h5ad")

In [3]:
adata_3 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/Article_SC_healthy_ensembl_id-postQC.h5ad")

In [5]:
rename_dict = {
    "male": "Male",     
    "female": "Female"
}
adata_3.obs["gender"] = adata_3.obs["gender"].replace(rename_dict)
adata_3.obs['gender'].value_counts()

/tmp/ipykernel_533004/1153549719.py:5: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata_3.obs["gender"] = adata_3.obs["gender"].replace(rename_dict)


gender
Male      39959
Female    14891
Name: count, dtype: int64

In [6]:
adata_3.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/Article_SC_healthy_ensembl_id-postQC.h5ad")###正确的做完dou和decot的post在1——perprocessing_measure里面

## adata_4-normalized

In [69]:
##把id加在了var列
adata_4 = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/statistic_data/HCL_AdultArtery_ensembl_id-premerge.h5ad")

In [70]:
adata_4

AnnData object with n_obs × n_vars = 9652 × 2875
    obs: 'batch', 'tissue', 'n_genes', 'n_counts', 'louvain', 'dataset', 'sample', 'age', 'gender', 'symptoms', 'intervention'
    var: 'n_cells', 'ensembl_id'
    uns: 'rank_genes_groups'
    obsm: 'X_tsne', 'X_umap'

In [71]:
# 查看数据的基本统计量
print("数据最小值:", adata_4.X.min())
print("数据最大值:", adata_4.X.max())
print("数据均值:", adata_4.X.mean())

数据最小值: 0.0
数据最大值: 8.012968
数据均值: 0.11366807


## adata_5 17554 15572

In [3]:
file_paths = [
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/HCA/GSE162950_GSM4968831.h5ad",
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/HCA/GSE162950_GSM4968832.h5ad",
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/HCA/GSE162950_GSM4968833.h5ad",
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/HCA/GSE162950_GSM4968834.h5ad",
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/HCA/GSE162950_GSM4968846.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]

In [4]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))

    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()

    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Sample1: 7021
Number of cells in sample Sample2: 1468
Number of cells in sample Sample3: 4108
Number of cells in sample Sample4: 3229
Number of cells in sample Sample5: 1728


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
adata_final

AnnData object with n_obs × n_vars = 17554 × 33694
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [9]:
# 创建从ensembl_id到gene_name的映射字典
ensembl_id_df = pd.read_csv("/home/lixiangyu/zr/Annotate/gene_names_to_ensembl_ALLFOUND_allfernandez_no6_withallslysz.csv")
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
adata_final.var['ensembl_id'] = adata_final.var_names
adata_final.var_names = [ensembl_to_gene[ensembl] if ensembl in ensembl_to_gene else ensembl 
                      for ensembl in adata_final.var_names]

In [11]:
adata_final.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-prepostQC.h5ad")

In [12]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_1627315/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [14]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-prepostQC.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-prepostQC_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Jan  9 17:11:13 2026 .. Analyzing cells in batch 'GSE162950_1'
Fri Jan  9 17:11:14 2026 .... Generating UMAP and estimating cell types
Fri Jan  9 17:11:44 2026 .... Estimating contamination
Fri Jan  9 17:11:49 2026 ...... Completed iteration: 10 | converge: 0.03111
Fri Jan  9 17:11:53 2026 ...... Completed iteration: 20 | converge: 0.01383
Fri Jan  9 17:11:57 2026 ...... Completed iteration: 30 | converge: 0.008659
Fri Jan  9 17:12:02 2026 ...... Completed iteration: 40 | converge: 0.006747
Fri Jan  9 17:12:06 2026 ...... Completed iteration: 50 | converge: 0.005603
Fri Jan  9 17:12:10 2026 ...... Completed iteration: 60 | converge: 0.004972
Fri Jan  9 17:12:15 2026 ...... Completed iteration: 70 | converge: 0.004377
Fri Jan  9 17:12:19 2026 ...... Completed iteration: 80 | converge: 0.003741
Fri Jan  9 17:12:23 2026 ...... Completed iteration: 90 | converge: 0.003

In [15]:
adata_final =sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-prepostQC_merged_postR.h5ad")

In [16]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [17]:
stats = adata_final.obs['pct_counts_mt'].describe()
print(stats)

count    17554.000000
mean         4.433153
std          6.213835
min          0.000000
25%          2.245062
50%          2.894905
75%          3.884881
max         90.822785
Name: pct_counts_mt, dtype: float64


In [18]:
adata_final.var

,ensembl_id,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
MIR1302-2HG,ENSG00000243485,False,1,0.000057,99.994303,1.0
FAM138A,ENSG00000237613,False,0,0.000000,100.000000,0.0
OR4F5,ENSG00000186092,False,0,0.000000,100.000000,0.0
AL627309.1,ENSG00000238009,False,62,0.003532,99.646804,62.0
RP11-34P13.8,ENSG00000239945,False,1,0.000057,99.994303,1.0
...,...,...,...,...,...,...
AC233755.2,ENSG00000277856,False,0,0.000000,100.000000,0.0
AC233755.1,ENSG00000275063,False,0,0.000000,100.000000,0.0
AC240274.1,ENSG00000271254,False,2825,0.206449,83.906802,3624.0
AC213203.2,ENSG00000277475,False,0,0.000000,100.000000,0.0


In [19]:
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

In [20]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=8000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=2) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 17554
Number of cells before counts filter: 17543
Number of cells beforeMT filter: 16406
Number of cells after MT filter: 15572


In [21]:
adata_final.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-postQC-R.h5ad")

In [22]:
adata_final.obs['sample'].value_counts()

sample
GSE162950_1    5936
GSE162950_3    3848
GSE162950_4    3092
GSE162950_5    1459
GSE162950_2    1237
Name: count, dtype: int64

In [23]:
adata_final = sc.read_h5ad("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-postQC-R.h5ad")

In [24]:
# 将obs中的'location'列重命名为'tissue'
adata_final.obs = adata_final.obs.rename(columns={'location': 'tissue'})

In [25]:
adata_final

AnnData object with n_obs × n_vars = 15572 × 25268
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'ensembl_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE162950_1_UMAP', 'decontX_GSE162950_2_UMAP', 'decontX_GSE162950_3_UMAP', 'decontX_GSE162950_4_UMAP', 'decontX_GSE162950_5_UMAP'
    layers: 'decontXcounts'

In [26]:
adata_final.write("/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-postQC-R.h5ad")

In [ ]:
####################################################################

## merge_healthy

In [18]:
file_paths = [
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/cellbygene_TS_Vasculature-postQC.h5ad",#adata_2
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/Article_SC_healthy_ensembl_id-postQC.h5ad",#adata_3
    "/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-postQC.h5ad"# adata_5
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3"
]

In [19]:
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Dataset1: 34918
Number of cells in sample Dataset2: 54850


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset3: 15572


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/merge.py:1337: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [20]:
adata_final.write("/home/lixiangyu/zr/Annotate/ANNOTATE_new/7_mapping_new_data/output_human_healthy/merged.h5ad")

In [21]:
adata_final

AnnData object with n_obs × n_vars = 105340 × 56038
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'tissue', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'

In [22]:
adata_final.obs['tissue'].value_counts()

tissue
Coronary artery         54850
aorta                   20682
Aorta                   14113
coronary artery         11778
thoracic aorta           2448
Vitelline Artery         1459
left coronary artery        6
abdominal aorta             4
Name: count, dtype: int64

In [25]:
adata_final.obs['tissue'] = adata_final.obs['tissue'].astype(str)
sample_specific_metadata = {
    'Coronary artery': {'tissue': 'Coronary Artery'},
    'aorta':{'tissue': 'Aorta'},
    'coronary artery': {'tissue':'Coronary Artery'},
    'thoracic aorta': {'tissue':'Thoracic Aorta'},
    'left coronary artery': {'tissue':'Left Coronary Artery'},
    'abdominal aorta':{'tissue':'Abdominal Aorta'}
}
# 应用样本特定的元数据
for sample, metadata in sample_specific_metadata.items():
    mask = adata_final.obs['tissue'] == sample
    for key, value in metadata.items():
        adata_final.obs.loc[mask, key] = value

In [26]:
adata_final.obs['tissue'].value_counts()

tissue
Coronary Artery         66628
Aorta                   34795
Thoracic Aorta           2448
Vitelline Artery         1459
Left Coronary Artery        6
Abdominal Aorta             4
Name: count, dtype: int64

In [27]:
adata_final.write("/home/lixiangyu/zr/Annotate/ANNOTATE_new/7_mapping_new_data/output_human_healthy/merged.h5ad")

# merged

## human_public(1)

In [34]:
file_paths = [
    "data/public_data/GSE143921/TAA_GSE143921_postQC.h5ad",#TAA
    "data/public_data/GSE155468/TAA_GSE155468_postQC.h5ad",#TAA
    "data/public_data/GSE166676_raw/AAA_GSE166676_postQC.h5ad",# AAA
    "data/public_data/GSE224587_raw/AAA_GSE224587_postQC.h5ad",# AAA
    "data/public_data/GSE226492_raw/AAA_GSE226492_postQC.h5ad",# AAA
    "data/public_data/GSE237230_raw/AAA_GSE237230_postQC.h5ad" ,# AAA
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3",
    "Dataset4",
    "Dataset5",
    "Dataset6"
]

In [ ]:
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset1: 6058


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset2: 44658


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset5: 134125


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset6: 5727


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [36]:
adata_final.write("output_data/public_data/new0509/merged.h5ad")

In [37]:
adata_final

AnnData object with n_obs × n_vars = 285484 × 86480
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'

## human_atlas(2)

In [38]:
file_paths = [
    "data/public_data/AS_atlas/GSE155512/GSE155512_postQC.h5ad",##AS_altas
    "data/public_data/AS_atlas/GSE159677/GSE159677_postQC.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSE179159_postQC.h5ad",
    "data/public_data/AS_atlas/GSE210152/GSE210152_postQC.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSE224273_postQC.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSE234077_postQC.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSE247238_postQC.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSE253904_postQC.h5ad",
    "data/public_data/AS_atlas/GSE131778/GSE131778_postQC.h5ad",
    "data/public_data/AS_atlas/GSE184073/GSE184073_postQC.h5ad",
    "data/public_data/AS_atlas/GSE196943/GSE196943_postQC.h5ad",
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3",
    "Dataset4",
    "Dataset5",
    "Dataset6",
    "Dataset7",
    "Dataset8",
    "Dataset9",
    "Dataset10",
    "Dataset11"

]

In [39]:
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset1: 8866


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset2: 44748


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset7: 8191


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells in sample Dataset8: 67835
Number of cells in sample Dataset9: 11501
Number of cells in sample Dataset10: 2371
Number of cells in sample Dataset11: 26330


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [40]:
adata_final.write("ANNOTATE_new/7_mapping_new_data/output_human_as_atlas/merged.h5ad")

In [41]:
adata_final

AnnData object with n_obs × n_vars = 302673 × 65957
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'patient'

# Merged all public human datas

In [33]:
file_paths = [
    "data/public_data/GSE143921/TAA_GSE143921_postQC-R.h5ad",#TAA
    "data/public_data/GSE155468/TAA_GSE155468_postQC-R.h5ad",#TAA
    "data/public_data/GSE166676_raw/AAA_GSE166676_postQC-R.h5ad",# AAA
    "data/public_data/GSE224587_raw/AAA_GSE224587_postQC-R.h5ad",# AAA
    "data/public_data/GSE226492_raw/AAA_GSE226492_postQC-R.h5ad",# AAA
    "data/public_data/GSE237230_raw/AAA_GSE237230_postQC-R.h5ad" ,# AAA
    
    "data/public_data/AS_atlas/GSE155512/GSE155512_postQC-R.h5ad",##AS_altas
    "data/public_data/AS_atlas/GSE159677/GSE159677_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE179159/GSE179159_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE210152/GSE210152_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE224273/GSE224273_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE234077/GSE234077_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE247238/GSE247238_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE253904/GSE253904_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE131778/GSE131778_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE184073/GSE184073_postQC-R.h5ad",
    "data/public_data/AS_atlas/GSE196943/GSE196943_postQC-R.h5ad",

    "/home/lixiangyu/zr/Annotate/data/human_normal_data/postQC/HCA_ensembl_id-postQC-R.h5ad"# adata_5
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3",
    "Dataset4",
    "Dataset5",
    "Dataset6",
    "Dataset7",
    "Dataset8",
    "Dataset9",
    "Dataset10",
    "Dataset11",
    "Dataset12",
    "Dataset13",
    "Dataset14",
    "Dataset15",
    "Dataset16",
    "Dataset17"
]

In [34]:
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

Number of cells in sample Dataset1: 6058
Number of cells in sample Dataset2: 47271
Number of cells in sample Dataset3: 12427
Number of cells in sample Dataset4: 82055
Number of cells in sample Dataset5: 96108
Number of cells in sample Dataset6: 4386
Number of cells in sample Dataset7: 8866
Number of cells in sample Dataset8: 45581
Number of cells in sample Dataset9: 20262
Number of cells in sample Dataset10: 30415
Number of cells in sample Dataset11: 9931
Number of cells in sample Dataset12: 66946
Number of cells in sample Dataset13: 4752
Number of cells in sample Dataset14: 77112
Number of cells in sample Dataset15: 11501
Number of cells in sample Dataset16: 1693
Number of cells in sample Dataset17: 26330


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [35]:
adata_final.write("/home/lixiangyu/zr/Annotate/ANNOTATE_new/7_mapping_new_data/output_all_human1226/merged.h5ad")
adata_final

AnnData object with n_obs × n_vars = 551694 × 119900
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'patient'
    obsm: 'decontX_GSE143921_1_UMAP', 'decontX_GSE143921_2_UMAP', 'decontX_GSE143921_3_UMAP', 'decontX_GSE143921_4_UMAP', 'decontX_GSE143921_5_UMAP', 'decontX_GSE143921_6_UMAP', 'decontX_GSE155468_10_UMAP', 'decontX_GSE155468_11_UMAP', 'decontX_GSE155468_1_UMAP', 'decontX_GSE155468_2_UMAP', 'decontX_GSE155468_3_UMAP', 'decontX_GSE155468_4_UMAP', 'decontX_GSE155468_5_UMAP', 'decontX_GSE155468_6_UMAP', 'decontX_GSE155468_7_UMAP', 'decontX_GSE155468_8_UMAP', 'decontX_GSE155468_9_UMAP', 'decontX_GSE166676_1_UMAP', 'decontX_GSE166676_2_UMAP', 'decontX_GSE166676_3_UMAP', 'decontX_GSE166676_4_UM